# Vector Search: Exact Search and the Approximate Indexes

The full code of [Vector Search: Exact Search and the Approximate Indexes](https://residual-stream.lvtan71.workers.dev/posts/vector-search-and-ann-indexes/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/vector-search-and-ann-indexes) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vector-search-and-ann-indexes.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" sentence-transformers faiss-cpu pyarrow huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "vector-search-and-ann-indexes"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `embed.py`

Embeds every sentence of WikiText-2's training set over 40 characters (71,865) and 1,000 sentences of its test set with all-MiniLM-L6-v2, normalized, and saves `docs.npy` and `queries.npy`. Its printed line was not kept, so it has no `.out`.

In [ ]:
!{sys.executable} embed.py

## `distances.py`

The ratio of the nearest to the farthest distance from a query, for random points in 2 to 384 dimensions and for the real sentence embeddings. Reads `docs.npy`, `queries.npy`.

In [ ]:
!{sys.executable} distances.py

The post printed:

```text
random points,   2 dimensions: nearest / farthest distance = 0.00
random points,  10 dimensions: nearest / farthest distance = 0.19
random points, 100 dimensions: nearest / farthest distance = 0.61
random points, 384 dimensions: nearest / farthest distance = 0.78
real embeddings,   384 dimensions: nearest / farthest distance = 0.63 (median of 200 queries)
```

## `recall_speed.py`

Recall@10, time per query and bytes per vector for exact search, IVF (256 and 1,024 lists), PQ, IVF-PQ with and without re-ranking, HNSW and random-hyperplane codes, one query at a time on one thread. Reads `docs.npy`, `queries.npy`.

In [ ]:
!{sys.executable} recall_speed.py

The post printed:

```text
L2 vs dot product, 100 queries: same top-10 sets: True; order differs at 4 positions, all between exactly tied scores (duplicate sentences): True

71,865 vectors, 384 dimensions:
flat (exact)                     recall@10  1.000    4.071 ms/query    1536 bytes/vector
  (IVF, 256 lists: built in 0.5 s)
IVF 256 lists, probe, 1          recall@10  0.496    0.026 ms/query    1544 bytes/vector
IVF 256 lists, probe, 4          recall@10  0.812    0.076 ms/query    1544 bytes/vector
IVF 256 lists, probe, 16         recall@10  0.955    0.279 ms/query    1544 bytes/vector
IVF 256 lists, probe, 64         recall@10  0.994    1.098 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 8         recall@10  0.799    0.062 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 16        recall@10  0.886    0.098 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 32        recall@10  0.941    0.174 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 64        recall@10  0.974    0.293 ms/query    1544 bytes/vector
  (PQ, 48 sub-vectors x 8 bits: built in 13.6 s)
PQ, 48-byte codes                recall@10  0.626    0.835 ms/query      48 bytes/vector
IVF-PQ, probe, 4                 recall@10  0.568    0.067 ms/query      56 bytes/vector
IVF-PQ, probe, 16                recall@10  0.604    0.118 ms/query      56 bytes/vector
IVF-PQ, probe, 64                recall@10  0.608    0.291 ms/query      56 bytes/vector
IVF-PQ + re-rank 100, probe, 16  recall@10  0.942    0.162 ms/query    1592 bytes/vector
IVF-PQ + re-rank 100, probe, 64  recall@10  0.969    0.328 ms/query    1592 bytes/vector
  (HNSW, M = 32: built in 47.4 s)
HNSW M=32, efSearch, 16          recall@10  0.876    0.089 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 32          recall@10  0.950    0.153 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 64          recall@10  0.983    0.279 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 128         recall@10  0.994    0.551 ms/query    1792 bytes/vector
LSH, 1024-bit codes, scanned     recall@10  0.631    0.364 ms/query     128 bytes/vector
```